# AMPS II · Week 10 — HW9
## Cohesive laws, brittleness, artificial compliance, the '+' side, debonding and an LLM-generated interface script

**Due before Week 11 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong. The whole HW runs in a few minutes.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: the exponential law, snap-back, springs in series | 15 |
| 2 | The bilinear law: the exact strip and the penalty $K_0$ | 14 |
| 3 | Brittleness: snap-back against the strip length | 13 |
| 4 | Springs on every facet: compliance against $h$ and $K$ | 15 |
| 5 | Counting the '+' sides | 14 |
| 6 | Where and when the inclusions debond | 14 |
| 7 | Debugging an LLM-generated interface script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run the cell once.

In [ ]:
%%writefile /tmp/fnx/czm.py
"""Cohesive interfaces, DOLFINx 0.11. Two subdomains (cell tags 1, 2), each with its own Lagrange space on a submesh, joined on the
interface Gamma by a traction-separation law T([[u]]), [[u]] = u2 - u1:
    sum_i int_{Omega_i} sigma(u_i) : eps(v_i) dx + int_Gamma T([[u]]) . [[v]] dS = lam int_{right edge} e_x . v ds
Laws (isotropic in the opening norm d = |[[u]]|): linear T = K [[u]];  exponential (the source) T = Gc/d0^2 exp(-d/d0) [[u]], d0 = Gc/(e sc);
bilinear: T = K0 [[u]] up to d = sc/K0, then the secant decreases to 0 at dc = 2 Gc / sc.
ORDERING: the interior-facet data are built explicitly with the cell of subdomain 1 as '+'. With plain meshtags DOLFINx orders the two cells
of a facet as it likes (it depends on the mesh and on the partition): u1('+') is then sometimes evaluated in a cell of subdomain 2."""
import gc
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem, LinearProblem
import dolfinx.fem.petsc as fp

comm = MPI.COMM_WORLD
# MPI + Python garbage collection: a problem object that is replaced (b = Bimaterial(...) in a loop) is freed by the cyclic garbage
# collector at a DIFFERENT moment on each rank; its __del__ destroys PETSc objects collectively -> one rank waits in the destroy, the
# other in the next assembly: a deadlock (seen here in about half of the -np 2 runs). Automatic collection is switched off and every
# constructor collects explicitly, i.e. at the same point on all ranks.
gc.disable()
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}

# ---------------------------------------------------------------- meshes and tags
def tag_cells(msh, f):
    """cell tags from midpoints: f(x) -> 1 or 2"""
    tdim = msh.topology.dim; nc = msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts
    mid = mesh.compute_midpoints(msh, tdim, np.arange(nc, dtype=np.int32))
    return mesh.meshtags(msh, tdim, np.arange(nc, dtype=np.int32), f(mid.T).astype(np.int32))

def strip_mesh(nx=8, ny=4, L=2.0, H=1.0):
    """[0, L] x [0, H] with quadrilaterals; subdomain 1: x < L/2, subdomain 2: x > L/2 (interface x = L/2, a mesh line if nx is even)"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], mesh.CellType.quadrilateral)
    return msh, tag_cells(msh, lambda x: np.where(x[0] < L / 2, 1, 2))

def plate_mesh(h, Lx=2.0, Ly=1.0, R=0.2, centres=((0.5, 0.5), (1.5, 0.5)), holes=False):
    """the source's plate with two circular inclusions (Gmsh fragment: the interface nodes are shared); tags 1 matrix, 2 inclusions.
    holes=True: the inclusions are removed (the fully debonded limit)"""
    import gmsh
    from dolfinx.io import gmsh as dg
    gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0); gmsh.model.add("plate")
    if comm.rank == 0:
        occ = gmsh.model.occ
        rect = occ.addRectangle(0, 0, 0, Lx, Ly); discs = [(2, occ.addDisk(cx, cy, 0, R, R)) for cx, cy in centres]
        (occ.cut if holes else occ.fragment)([(2, rect)], discs); occ.synchronize()
        surfs = [t for _, t in gmsh.model.getEntities(2)]
        mat = [t for t in surfs if gmsh.model.isInside(2, t, [0.0, 0.0, 0.0])]
        gmsh.model.addPhysicalGroup(2, mat, 1)
        if not holes:
            gmsh.model.addPhysicalGroup(2, [t for t in surfs if t not in mat], 2)
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", h); gmsh.option.setNumber("Mesh.CharacteristicLengthMin", h)
        gmsh.model.mesh.generate(2)
    # interior-facet integrals (the interface, DG coupling) need the cells on BOTH sides of a facet on the rank that owns it:
    # Gmsh meshes are created without ghost cells by default -> facets on a partition boundary would be silently skipped
    md = dg.model_to_mesh(gmsh.model, comm, 0, gdim=2, partitioner=mesh.create_cell_partitioner(mesh.GhostMode.shared_facet, 2)); gmsh.finalize()
    return md.mesh, md.cell_tags

def interface_data(msh, ct, a=1, b=2):
    """owned interior facets between cells tagged a and b as integration data [cell_a, local facet, cell_b, local facet, ...]: '+' = side a.
    Also returns the facet indices."""
    tdim = msh.topology.dim; fdim = tdim - 1
    msh.topology.create_connectivity(fdim, tdim); msh.topology.create_connectivity(tdim, fdim)
    f2c, c2f = msh.topology.connectivity(fdim, tdim), msh.topology.connectivity(tdim, fdim)
    tag = np.zeros(msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts, dtype=np.int32)
    tag[ct.indices] = ct.values
    data, facets = [], []
    for f in range(msh.topology.index_map(fdim).size_local):
        cs = f2c.links(f)
        if len(cs) == 2 and {tag[cs[0]], tag[cs[1]]} == {a, b}:
            c0, c1 = (cs[0], cs[1]) if tag[cs[0]] == a else (cs[1], cs[0])
            data += [c0, int(np.where(c2f.links(c0) == f)[0][0]), c1, int(np.where(c2f.links(c1) == f)[0][0])]
            facets.append(f)
    return np.array(data, dtype=np.int32), np.array(facets, dtype=np.int32)

# ---------------------------------------------------------------- 1D references
def T_exp(d, Gc, sc):
    """exponential law (normal opening d >= 0): T = Gc/d0^2 d exp(-d/d0), peak sc at d = d0 = Gc/(e sc), integral Gc"""
    d0 = Gc / (np.e * sc)
    return Gc / d0**2 * d * np.exp(-d / d0)

def strip_exact(d, L, E, Gc, sc):
    """bar/strip of length L (uniaxial stress) with one interface: sigma = T(d), end displacement u = sigma L / E + d"""
    s = T_exp(d, Gc, sc)
    return s, s * L / E + d

# ---------------------------------------------------------------- the interface model
class Bimaterial:
    count = 0
    def __init__(self, msh, ct, E=(10.0, 70.0), nu=(0.3, 0.2), law="exp", K=None, Gc=0.01, sc=0.5, deg=1, plane="stress", order="explicit"):
        """E, nu: (subdomain 1, subdomain 2); law "linear" (K), "exp" (Gc, sc), "bilinear" (K, Gc, sc); order "explicit" or "meshtags".
        The laws are isotropic in |[[u]]| (as in the source): they soften in compression too — no contact, interpenetration is possible."""
        collect()
        self.msh, self.ct = msh, ct; tdim = msh.topology.dim; fdim = tdim - 1
        subs = [mesh.create_submesh(msh, tdim, ct.find(i)) for i in (1, 2)]
        self.sub, self.em = [s[0] for s in subs], [s[1] for s in subs]
        self.V = [fem.functionspace(s, ("Lagrange", deg, (2,))) for s in self.sub]
        self.u = [fem.Function(V) for V in self.V]; self.w = [ufl.TestFunction(V) for V in self.V]
        data, self.ifacets = interface_data(msh, ct)
        if order == "explicit":
            self.dS = ufl.Measure("dS", domain=msh, subdomain_data=[(1, data)])(1)
        else:
            fs = np.sort(self.ifacets)
            self.dS = ufl.Measure("dS", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, fs, np.ones(len(fs), dtype=np.int32)))(1)
        self.dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
        def lame(Ei, ni):
            lam, mu = Ei * ni / ((1 + ni) * (1 - 2 * ni)), Ei / (2 * (1 + ni))
            return (2 * lam * mu / (lam + 2 * mu) if plane == "stress" else lam), mu
        self.mat = [lame(E[i], nu[i]) for i in (0, 1)]
        self.sig = lambda u, i: self.mat[i][0] * ufl.tr(ufl.sym(ufl.grad(u))) * ufl.Identity(2) + 2 * self.mat[i][1] * ufl.sym(ufl.grad(u))
        self.jmp = self.u[1]("-") - self.u[0]("+")
        self.T = self.law(self.jmp, law, K, Gc, sc)
        n1 = ufl.FacetNormal(msh)("+")                                 # outward normal of side 1: the normal gap is [[u]] . n1
        self.gap = ufl.dot(self.jmp, n1)
        self.lam = fem.Constant(msh, 0.0)
        self.Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
        right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.Lx))
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, np.sort(right), np.ones(len(right), dtype=np.int32)))(1)
        self.F = [ufl.inner(self.sig(self.u[0], 0), ufl.sym(ufl.grad(self.w[0]))) * self.dx(1) - ufl.dot(self.T, self.w[0]("+")) * self.dS,
                  ufl.inner(self.sig(self.u[1], 1), ufl.sym(ufl.grad(self.w[1]))) * self.dx(2) + ufl.dot(self.T, self.w[1]("-")) * self.dS]
        self.bcs = []; self.n = ufl.FacetNormal(msh)
        self.glen = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * self.dS)), op=MPI.SUM)

    def law(self, j, law, K, Gc, sc):
        d = ufl.sqrt(ufl.dot(j, j) + 1e-30)
        if law == "linear":
            return K * j
        if law == "exp":
            self.d0 = d0 = Gc / (np.e * sc)
            return Gc / d0**2 * ufl.exp(-d / d0) * j
        if law == "bilinear":                                         # secant K0 up to d1 = sc/K0, then sc (dc - d)/(dc - d1)/d, 0 beyond dc
            d1, dc = sc / K, 2 * Gc / sc; self.d0 = d1
            sec = ufl.conditional(ufl.lt(d, d1), K, ufl.conditional(ufl.lt(d, dc), sc * (dc - d) / ((dc - d1) * d), 0.0))
            return sec * j
        raise ValueError(law)

    def fix(self, i, marker, comp):
        """u_comp = 0 on the boundary facets of subdomain i (0 or 1) selected by marker"""
        V = self.V[i]; Vc, _ = V.sub(comp).collapse(); fdim = self.sub[i].topology.dim - 1
        f = mesh.locate_entities_boundary(self.sub[i], fdim, marker)
        self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(comp), Vc), fdim, f), V.sub(comp)))

    def fix_point(self, i, marker, comp):
        V = self.V[i]; Vc, _ = V.sub(comp).collapse()
        self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_geometrical((V.sub(comp), Vc), marker), V.sub(comp)))

    def setup(self, load_side=0, control="traction"):
        """the right edge belongs to subdomain load_side (0 or 1); control "traction" (lam = traction) or "displacement" (lam = u_x there)"""
        i = self.load_side = load_side; self.control = control
        F = list(self.F)
        if control == "traction":
            F[i] = F[i] - self.lam * self.w[i][0] * self.ds
        else:
            V = self.V[i]; Vc, _ = V.sub(0).collapse(); fdim = self.sub[i].topology.dim - 1
            self.ubc = fem.Function(Vc)
            f = mesh.locate_entities_boundary(self.sub[i], fdim, lambda x: np.isclose(x[0], self.Lx))
            self.bcs.append(fem.dirichletbc(self.ubc, fem.locate_dofs_topological((V.sub(0), Vc), fdim, f), V.sub(0)))
        Bimaterial.count += 1
        self.pb = NonlinearProblem(F, self.u, bcs=self.bcs, petsc_options_prefix=f"bm{Bimaterial.count}_", entity_maps=self.em, kind="mpi",
                                   petsc_options={**LU, "snes_type": "newtonls", "snes_linesearch_type": "none", "snes_max_it": 30,
                                                  "snes_rtol": 1e-10, "snes_atol": 1e-12})
        n = self.n          # opening functional g(u) = mean normal opening (1/|Gamma|) int (u2 - u1) . n1 dS, n1 = n('+') (outward normal of side 1)
        self.Gm = [fem.form(-ufl.dot(self.w[0]("+"), n("+")) * self.dS, entity_maps=self.em),
                   fem.form(ufl.dot(self.w[1]("-"), n("+")) * self.dS, entity_maps=self.em)]
        self.f_open = fem.form(ufl.dot(self.jmp, n("+")) * self.dS, entity_maps=self.em)
        self.f_ux = fem.form(self.u[i][0] * self.ds, entity_maps=self.em); self.f_len = fem.form(1.0 * self.ds)
        self.f_reac = fem.form(ufl.dot(ufl.dot(self.sig(self.u[i], i), n), ufl.as_vector([1.0, 0.0])) * self.ds, entity_maps=self.em)

    def solve_load(self, value):
        """load (traction) or displacement control: Newton (SNES) at a fixed lam. Returns (SNES reason, iterations)."""
        if self.control == "traction":
            self.lam.value = value
        else:
            self.ubc.x.array[:] = value
        self.pb.solve()
        return self.pb.solver.getConvergedReason(), self.pb.solver.getIterationNumber()

    def solve_opening(self, target, tol=1e-9, maxit=30):
        """opening control (traction loading): solve R(u) - lam f = 0 together with g(u) = target, a bordered Newton step with two
        right-hand sides per iteration (K a = -R, K b = f, dlam from the constraint). Returns (iterations, |R|)."""
        pb = self.pb; x = pb.x; A = pb.A; ksp = pb.solver.getKSP()
        fp.assign(self.u, x)
        g = fp.assemble_vector(self.Gm, kind="mpi"); g.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); g.scale(1.0 / self.glen)
        R, R1 = pb.b.duplicate(), pb.b.duplicate(); r0 = None
        for it in range(maxit):
            lam0 = float(self.lam.value)
            pb.solver.computeFunction(x, R)
            self.lam.value = lam0 + 1.0; pb.solver.computeFunction(x, R1); self.lam.value = lam0
            f = R.copy(); f.axpy(-1.0, R1)                             # f = dR/d(-lam): the load vector
            nr = R.norm(); r0 = r0 or max(nr, 1e-30); res = target - g.dot(x)
            if it > 0 and nr < tol * max(r0, 1.0) and abs(res) < tol * max(abs(target), 1e-12):
                break
            pb.solver.computeJacobian(x, A, A); ksp.setOperators(A)
            a, b = R.duplicate(), R.duplicate(); mR = R.copy(); mR.scale(-1.0)
            ksp.solve(mR, a); ksp.solve(f, b)
            dl = (res - g.dot(a)) / g.dot(b)
            x.axpy(1.0, a); x.axpy(dl, b); self.lam.value = lam0 + dl
        fp.assign(x, self.u)
        for u in self.u:
            u.x.scatter_forward()
        return it, nr

    def opening(self):
        """mean normal opening of the interface"""
        return comm.allreduce(fem.assemble_scalar(self.f_open), op=MPI.SUM) / self.glen

    def end_disp(self):
        """mean u_x on the loaded (right) edge"""
        return comm.allreduce(fem.assemble_scalar(self.f_ux), op=MPI.SUM) / comm.allreduce(fem.assemble_scalar(self.f_len), op=MPI.SUM)

    def force(self):
        """force on the right edge: lam * height (traction control) or the integral of sigma_xx (displacement control)"""
        if self.control == "traction":
            return float(self.lam.value) * comm.allreduce(fem.assemble_scalar(self.f_len), op=MPI.SUM)
        return comm.allreduce(fem.assemble_scalar(self.f_reac), op=MPI.SUM)

    def interface_jumps(self):
        """(facet midpoints, [[u]] there) of all interface facets, gathered on rank 0 (point evaluation in each submesh)"""
        from dolfinx import geometry
        mid = mesh.compute_midpoints(self.msh, self.msh.topology.dim - 1, self.ifacets)
        allmid = comm.allgather(mid); P = np.vstack(allmid) if sum(len(m) for m in allmid) else np.zeros((0, 3))
        vals = []
        for k in (0, 1):
            m = self.sub[k]; tdim = m.topology.dim; nloc = m.topology.index_map(tdim).size_local
            cand = geometry.compute_colliding_cells(m, geometry.compute_collisions_points(geometry.bb_tree(m, tdim), P), P)
            out = np.zeros((len(P), 2)); cnt = np.zeros(len(P))
            for j in range(len(P)):
                own = [c for c in cand.links(j) if c < nloc]
                if own:
                    out[j] = np.asarray(self.u[k].eval(P[j], [own[0]])).reshape(-1); cnt[j] = 1
            out = comm.allreduce(out, op=MPI.SUM); cnt = comm.allreduce(cnt, op=MPI.SUM)
            vals.append(out / np.maximum(cnt, 1)[:, None])
        return P, vals[1] - vals[0]

# ---------------------------------------------------------------- references
def bonded_plate(h, E=(10.0, 70.0), nu=(0.3, 0.2), holes=False, load=0.1, deg=2):
    """single CG mesh (perfect bond) or the plate with holes; left u_x = 0, bottom u_y = 0, traction load * e_x on the right (plane stress).
    Returns force / mean end displacement."""
    collect()
    msh, ct = plate_mesh(h, holes=holes)
    Q = fem.functionspace(msh, ("DG", 0)); Ef, nf = fem.Function(Q), fem.Function(Q)
    Ef.x.array[:] = E[0]; nf.x.array[:] = nu[0]
    if not holes:
        Ef.x.array[ct.find(2)] = E[1]; nf.x.array[ct.find(2)] = nu[1]
    lam, mu = Ef * nf / (1 - nf**2), Ef / 2 / (1 + nf)
    V = fem.functionspace(msh, ("Lagrange", deg, (2,))); u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    sig = lambda w: lam * ufl.tr(ufl.sym(ufl.grad(w))) * ufl.Identity(2) + 2 * mu * ufl.sym(ufl.grad(w))
    fdim = 1; V0, _ = V.sub(0).collapse(); V1, _ = V.sub(1).collapse()
    bcs = [fem.dirichletbc(fem.Function(V0), fem.locate_dofs_topological((V.sub(0), V0), fdim, mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))), V.sub(0)),
           fem.dirichletbc(fem.Function(V1), fem.locate_dofs_topological((V.sub(1), V1), fdim, mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))), V.sub(1))]
    Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
    right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], Lx))
    ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, np.sort(right), np.ones(len(right), dtype=np.int32)))(1)
    Bimaterial.count += 1
    uh = LinearProblem(ufl.inner(sig(u), ufl.sym(ufl.grad(v))) * ufl.dx, load * v[0] * ds, bcs=bcs, petsc_options=LU,
                       petsc_options_prefix=f"bond{Bimaterial.count}_").solve()
    H = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ds)), op=MPI.SUM)
    ub = comm.allreduce(fem.assemble_scalar(fem.form(uh[0] * ds)), op=MPI.SUM) / H
    return load * H / ub

def dg_intrinsic(msh, ct, K_int, K_bulk, E=(10.0, 70.0), nu=(0.3, 0.2), beta=1e3, u_app=0.002):
    """the source's model, linearised: DG1 everywhere, a cohesive spring on EVERY interior facet (K_int on the interface, K_bulk elsewhere),
    Nitsche for the Dirichlet conditions (left u_x = u_y = 0, bottom u_y = 0, right u_x = u_app). Returns reaction force / u_app."""
    collect()
    tdim = msh.topology.dim; fdim = tdim - 1
    Q = fem.functionspace(msh, ("DG", 0)); Ef, nf = fem.Function(Q), fem.Function(Q)
    Ef.x.array[:] = E[0]; nf.x.array[:] = nu[0]; Ef.x.array[ct.find(2)] = E[1]; nf.x.array[ct.find(2)] = nu[1]
    lam, mu = Ef * nf / (1 - nf**2), Ef / 2 / (1 + nf)
    V = fem.functionspace(msh, ("DG", 1, (2,))); u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    sig = lambda w: lam * ufl.tr(ufl.sym(ufl.grad(w))) * ufl.Identity(2) + 2 * mu * ufl.sym(ufl.grad(w))
    data, ifac = interface_data(msh, ct)
    msh.topology.create_connectivity(fdim, tdim)
    allint = np.arange(msh.topology.index_map(fdim).size_local, dtype=np.int32)
    nb = np.array([len(msh.topology.connectivity(fdim, tdim).links(f)) for f in allint]); allint = allint[nb == 2]
    tags = np.where(np.isin(allint, ifac), 1, 2).astype(np.int32)
    dS = ufl.Measure("dS", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, allint, tags))
    Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
    fl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0)); fr = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], Lx))
    fb = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))
    idx = np.concatenate([fl, fr, fb]); val = np.concatenate([np.full(len(fl), 1), np.full(len(fr), 2), np.full(len(fb), 3)]).astype(np.int32)
    o = np.argsort(idx); ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, idx[o], val[o]))
    n = ufl.FacetNormal(msh); h = ufl.CellDiameter(msh); ex, ey = ufl.as_vector([1.0, 0.0]), ufl.as_vector([0.0, 1.0])
    def nit(c, g, tag):                                               # Nitsche for the component c (unit vector) = g on ds(tag)
        return (-ufl.dot(ufl.dot(sig(u), n), c) * ufl.dot(v, c) - ufl.dot(ufl.dot(sig(v), n), c) * ufl.dot(u, c)
                + beta / h * ufl.dot(u, c) * ufl.dot(v, c)) * ds(tag), (-ufl.dot(ufl.dot(sig(v), n), c) * g + beta / h * g * ufl.dot(v, c)) * ds(tag)
    a = ufl.inner(sig(u), ufl.sym(ufl.grad(v))) * ufl.dx + K_int * ufl.dot(ufl.jump(u), ufl.jump(v)) * dS(1) + K_bulk * ufl.dot(ufl.jump(u), ufl.jump(v)) * dS(2)
    L = ufl.dot(fem.Constant(msh, np.zeros(2)), v) * ufl.dx
    for c, g, tag in ((ex, 0.0, 1), (ey, 0.0, 1), (ex, u_app, 2), (ey, 0.0, 3)):
        aa, ll = nit(c, g, tag); a += aa; L += ll
    Bimaterial.count += 1
    uh = LinearProblem(a, L, petsc_options=LU, petsc_options_prefix=f"dg{Bimaterial.count}_").solve()
    F = comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(ufl.dot(sig(uh), n), ex) * ds(2))), op=MPI.SUM)
    return F / u_app

## Q1 — Theory (15 pts)

**(a) (5)** For $T(\delta)=\frac{G_c}{\delta_0^2}\delta e^{-\delta/\delta_0}$ show that the maximum is at $\delta=\delta_0$ with $T=G_c/(e\delta_0)$, that $\int_0^\infty T\,d\delta=G_c$, and that $\min T'=-G_ce^{-2}/\delta_0^2$ at $\delta=2\delta_0$.
Hence $\delta_0=G_c/(e\sigma_c)$ and the initial stiffness is $e^2\sigma_c^2/G_c$.

**(b) (5)** A bar of length $L$ (modulus $E$) with one interface obeying any law $T(\delta)$: derive $u(\delta)$, the snap-back condition, and $L_c$ for the exponential law. Express $L_c$ with $E$, $G_c$, $\sigma_c$ only.

**(c) (5)** A 1D chain: $n$ cells of length $h$ ($nh=L$, modulus $E$) separated by springs of stiffness $K$ (per unit area) at every cell boundary. Show $E_{\rm eff}=E/(1+E/(Kh))$. What does this predict for the
source's model as $h$ is halved, and what must $Kh/E$ be for a 1 % error?

> *your answer*

## Q2 — The bilinear law: the exact strip and the penalty $K_0$ (14 pts)

`law="bilinear"`: $T=K_0\delta$ up to $\delta_1=\sigma_c/K_0$, then linearly down to zero at $\delta_c=2G_c/\sigma_c$.

**(a) (4)** Write the exact strip response $u(\delta)$ and the snap-back condition for this law.

**(b) (6)** Run the strip ($L=2$, $E=10$, $G_c=0.01$, $\sigma_c=0.5$) with $K_0=10^3$ under opening control up to $\delta_c$ and compare with (a) (maximum deviation). Note: the law has a kink at $\delta_1$ and a second one at $\delta_c$.

**(c) (4)** Repeat with $K_0=10^2$ and $10^4$: the end displacement at the peak and $L_c$. How does the penalty enter the physics?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b)(c) — bilinear strip against the exact response, K0 = 1e2, 1e3, 1e4

> *your answer (Q2)*

## Q3 — Brittleness: snap-back against the strip length (13 pts)

**(a) (8)** For the exponential law, run the strip under opening control for $L$ = 0.2, 0.4, 0.8, 2 (keep 4 cells per unit length) and determine numerically whether $u$ decreases anywhere after the peak
(the minimum of $\Delta u/\Delta\delta$ along the path). Compare with $L_c=0.40$.

**(b) (5)** Which of these tests could have been run with displacement control? What does that mean for a laboratory test of a long specimen?

In [ ]:
%%fenicsx -np 2
# TODO: Q3 — snap-back or not for L = 0.2, 0.4, 0.8, 2

> *your answer (Q3)*

## Q4 — Springs on every facet: compliance against $h$ and $K$ (15 pts)

`dg_intrinsic(msh, ct, K_int, K_bulk)` is the source's model, linearised (DG1, a spring on every interior facet, Nitsche supports, left clamped). Use the same model with almost rigid springs
($10^6K_s$) on the same mesh as the bonded reference $k_{\rm bond}(h)$ (the perfectly bonded CG plate with these supports gives 5.95).

**(a) (8)** With $K_{\rm int}=K_{\rm bulk}=K$, compute the stiffness for $h$ = 0.12, 0.06, 0.03 and $K/K_s$ = 1, 10, 100 ($K_s=295$, the source's bulk value). Plot $1/k-1/k_{\rm bond}$ against $1/(Kh)$.

**(b) (7)** Is the added compliance proportional to $1/(Kh)$ as Q1(c) predicts? Estimate the constant and the $K$ needed for 1 % at $h=0.03$.

In [ ]:
%%fenicsx -np 2
# TODO: Q4 — the DG model for h = 0.12, 0.06, 0.03 and K = Ks, 10 Ks, 100 Ks

In [ ]:
# TODO: Q4(a) plot

> *your answer (Q4)*

## Q5 — Counting the '+' sides (14 pts)

`dolfinx.fem.compute_integration_domains(IntegralType.interior_facet, msh.topology, facets)` returns the data DOLFINx uses for plain meshtags: `[cell, local facet, cell, local facet]` per facet, the first
cell being '+'.

**(a) (8)** For the plate mesh ($h=0.06$), count the interface facets whose first cell lies in the inclusion, with `-np 1` and with `-np 2` (run the cell twice). Do the same for `interface_data`.

**(b) (6)** A colleague replaces `u1("+")` by `u1("-")` wherever the inclusion comes first — "just flip the sign". Why does that not work in a single form, and what are the two correct remedies?

In [ ]:
%%fenicsx -np 2
# TODO: Q5(a) — reversed facets with the default ordering and with interface_data (-np 1 and -np 2)

> *your answer (Q5)*

## Q6 — Where and when the inclusions debond (14 pts)

**(a) (7)** With a stiff linear interface ($K=10^6$, i.e. perfect bond) and the traction $\lambda=1$ on the right, compute the normal interface traction $t_n=K\,[\![\mathbf u]\!]\cdot\mathbf n$ at the interface facets
(`interface_jumps`; the outward normal of the inclusion is $(\mathbf x-\mathbf c)/R$) and find its maximum and the angle where it occurs. Predict the load $\lambda$ at which the exponential interface
reaches $\sigma_c$ first.

**(b) (7)** Compare with the CZM plate under opening control: at which force does the curve leave the initial line by 1 %? Why is that force not the same as your prediction? (Hint: the initial stiffness
of the law, and the peak is at $\delta_0$ — not where softening starts.)

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — normal traction of the bonded interface, predicted onset, and the CZM curve

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated interface script (15 pts)

A student asked an LLM for "a DOLFINx 0.11 script for the delamination of a coating: a substrate and a coating joined by an exponential cohesive zone, plane stress; report the interface strength and verify it".
The answer is below, as received.

**(a) (3)** Run it with `-np 2`. What does it print, and does the "verification" test anything?

**(b) (8)** Find the defects (there are at least four, some silent). For each: what is wrong, what it does to the result (measure it by fixing the others), and which test reveals it.

**(c) (4)** Write the corrected script (or use `czm.py`) and verify the strength with tests that do not need the answer: the law in NumPy, the exact strip, `-np 1` vs `-np 2`.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): delamination of a coating — two layers joined by an exponential cohesive interface, plane stress
import numpy as np, ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
comm = MPI.COMM_WORLD
E, nu, Gc, sc, L = 10.0, 0.3, 0.01, 0.5, 2.0
d0 = Gc / sc                                                         # characteristic opening
msh = mesh.create_rectangle(comm, [[0, 0], [L, 1]], [8, 4], mesh.CellType.quadrilateral)
c1 = mesh.locate_entities(msh, 2, lambda x: x[0] <= L / 2 + 1e-12)     # substrate cells
c2 = mesh.locate_entities(msh, 2, lambda x: x[0] >= L / 2 - 1e-12)     # coating cells
s1, e1, _, _ = mesh.create_submesh(msh, 2, c1); s2, e2, _, _ = mesh.create_submesh(msh, 2, c2)
nc = msh.topology.index_map(2).size_local + msh.topology.index_map(2).num_ghosts
tag = np.full(nc, 2, dtype=np.int32); tag[c1] = 1; ct = mesh.meshtags(msh, 2, np.arange(nc, dtype=np.int32), tag)
fi = mesh.locate_entities(msh, 1, lambda x: np.isclose(x[0], L / 2))
dS = ufl.Measure("dS", domain=msh, subdomain_data=mesh.meshtags(msh, 1, np.sort(fi), np.ones(len(fi), dtype=np.int32)))(1)
dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
lam, mu = E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))       # Lame constants
V1 = fem.functionspace(s1, ("Lagrange", 1, (2,))); V2 = fem.functionspace(s2, ("Lagrange", 1, (2,)))
u1, u2 = fem.Function(V1), fem.Function(V2); w1, w2 = ufl.TestFunction(V1), ufl.TestFunction(V2)
sig = lambda u: lam * ufl.tr(ufl.sym(ufl.grad(u))) * ufl.Identity(2) + 2 * mu * ufl.sym(ufl.grad(u))
j = u2("-") - u1("+"); d = ufl.sqrt(ufl.dot(j, j) + 1e-30)
T = Gc / d0**2 * ufl.exp(-d / d0) * j
t = fem.Constant(msh, 0.0)
right = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], L))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, np.sort(right), np.ones(len(right), dtype=np.int32)))(1)
F = [ufl.inner(sig(u1), ufl.sym(ufl.grad(w1))) * dx(1) - ufl.dot(T, w1("+")) * dS,
     ufl.inner(sig(u2), ufl.sym(ufl.grad(w2))) * dx(2) + ufl.dot(T, w2("-")) * dS - t * w2[0] * ds]
V1x, _ = V1.sub(0).collapse(); V1y, _ = V1.sub(1).collapse()
bcs = [fem.dirichletbc(fem.Function(V1x), fem.locate_dofs_topological((V1.sub(0), V1x), 1, mesh.locate_entities_boundary(s1, 1, lambda x: np.isclose(x[0], 0))), V1.sub(0)),
       fem.dirichletbc(fem.Function(V1y), fem.locate_dofs_geometrical((V1.sub(1), V1y), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), V1.sub(1))]
pb = NonlinearProblem(F, [u1, u2], bcs=bcs, entity_maps=[e1, e2], kind="mpi", petsc_options_prefix="llm_",
                      petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "none", "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
strength = 0.0
for tv in np.linspace(0.01, 0.6, 60):                                 # load control up to beyond the strength
    t.value = tv; pb.solve()
    if pb.solver.getConvergedReason() > 0:
        strength = tv
area2 = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=s2))), op=MPI.SUM)
print(f"interface strength = {strength:.3f} (sigma_c = {sc}); coating area {area2:.3f} -> verified")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …